# 5.7 · 随机森林分类器 / Random Forest Classifier

> **课程定位 / Where this fits**
> 5.6 结尾发现: 单棵满树**高方差**。随机森林的想法极简——**训练很多棵去相关的树, 投票平均掉方差**。它几乎开箱即用、不怕过拟合、自带特征重要性和 OOB 评估, 是 Kaggle/工业界最常用的强基线之一。
> Random Forest = many de-correlated trees, voting to average away variance. Near-zero-tuning, hard to overfit, the go-to strong baseline.

> 💡 **面试相关 / Interview-relevant**
> - "bagging 为什么能降方差 / 数学原理" ★★★★★
> - "随机森林的'随机'体现在哪两处" ★★★★★（样本 bootstrap + 特征子集）
> - "为什么要特征随机(只 bootstrap 不够)" ★★★★★（去相关）
> - "OOB 误差是什么" ★★★★
> - "RF vs 单树 / RF 会过拟合吗" ★★★★
> - "RF vs GBDT 区别" ★★★★（并行bagging vs 串行boosting）

---

## 学习目标 / Learning Objectives
1. bagging 降方差的数学(去相关的平均)。
2. 随机森林的两处随机: **样本 bootstrap + 特征子集**。
3. **OOB** 免费验证。
4. 树的数量 / `max_features` 的影响。
5. RF vs 单树 vs 预告 GBDT。

## 目录 / TOC
1. [bagging 降方差 ⭐](#1)
2. [两处随机: 去相关 ⭐](#2)
3. [🚢 数据 + 单树 vs 森林](#3)
4. [OOB 免费验证 ⭐](#4)
5. [调参: n_estimators / max_features](#5)
6. [特征重要性 + RF vs GBDT](#6)
7. [小结](#7)


<a id="1"></a>
## 1. bagging 降方差 ⭐ / Why Bagging Reduces Variance

**Bagging** = Bootstrap AGGregating: 对训练集**有放回抽样**得 B 个 bootstrap 子集, 每个训一棵树, 预测时**投票/平均**。

**数学**: B 个方差均为 $\sigma^2$、两两相关系数 $\rho$ 的预测, 平均后方差：
$$\text{Var}\Big(\tfrac1B\sum_i\hat f_i\Big) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$$

- $B\to\infty$: 第二项 $\to0$, 剩 $\rho\sigma^2$。
- **关键**: 只增加 B 不够——下限被相关性 $\rho$ 卡住。**必须降低 $\rho$(让树彼此不同)**。这就是为什么光 bootstrap 不够, 还要**特征随机**(下一节)。
偏差基本不变, 方差大降 → 这是 bagging 的核心。


<a id="2"></a>
## 2. 两处随机: 去相关 ⭐ / Two Sources of Randomness

随机森林 = bagging 树 + **每次分裂只考虑随机的特征子集**:
1. **样本随机**: 每棵树用一个 bootstrap 样本(约 63% 不重复样本)。
2. **特征随机**: 每个分裂点只从随机 $m$ 个特征(分类默认 $m=\sqrt{d}$)里选最优。

为什么要第 2 点: 若有一个超强特征(如 Titanic 的 sex), 光 bootstrap 的话每棵树都先按它分裂 → 树高度相似($\rho$ 大)→ 平均效果差。强制随机特征**逼不同树看不同特征 → 去相关 → $\rho$ 降 → 方差降更多**。


<a id="3"></a>
## 3. 数据 + 单树 vs 森林 / Single Tree vs Forest

复用 **Titanic**(5.6 介绍并清洗过), 同样的特征。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

df = sns.load_dataset("titanic")
feat = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
d = df[feat + ["survived"]].copy()
d["age"] = d["age"].fillna(d["age"].median())
d["fare"] = d["fare"].fillna(d["fare"].median())
d["sex"] = (d["sex"] == "male").astype(int)
from sklearn.model_selection import train_test_split, cross_val_score
X, y = d[feat].values, d["survived"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
tree = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(X_tr, y_tr)
print(f"单棵满树   test 准确率: {tree.score(X_te, y_te):.3f}")
print(f"随机森林   test 准确率: {rf.score(X_te, y_te):.3f}")
# CV 方差对比 / variance via CV std
print(f"单树 5-fold CV: {cross_val_score(DecisionTreeClassifier(random_state=0),X,y,cv=5).std():.3f} (std)")
print(f"森林 5-fold CV: {cross_val_score(RandomForestClassifier(100,random_state=0),X,y,cv=5).std():.3f} (std, 更稳)")


<a id="4"></a>
## 4. OOB 免费验证 ⭐ / Out-of-Bag Estimate

每棵树的 bootstrap 只用了约 63% 样本, **剩下约 37% 是"袋外(OOB)"样本**, 该树没见过。用每个样本"没见过它的那些树"来预测它 → 得到一个**几乎免费的验证误差**, 无需单独留验证集。


In [ ]:
rf_oob = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0).fit(X_tr, y_tr)
print(f"OOB 准确率:  {rf_oob.oob_score_:.3f}")
print(f"测试准确率:  {rf_oob.score(X_te, y_te):.3f}")
print("两者接近 → OOB 是可靠的免费验证, 小数据尤其有用(省下留出集)")
# 验证 37% 规则 / verify ~37% out-of-bag
n = 10000; rng = np.random.default_rng(0)
sampled = np.unique(rng.integers(0, n, n))
print(f"\nbootstrap 覆盖率: {len(sampled)/n:.1%} (理论 1-1/e≈63.2%), OOB≈36.8%")


<a id="5"></a>
## 5. 调参: n_estimators / max_features / Tuning


In [ ]:
# 树越多越稳(不会过拟合, 只会收敛) / more trees -> converge, never overfit
ns = [1, 5, 10, 25, 50, 100, 200, 400]
accs = [RandomForestClassifier(n, random_state=0).fit(X_tr,y_tr).score(X_te,y_te) for n in ns]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(ns, accs, "o-"); axes[0].set_xlabel("n_estimators"); axes[0].set_ylabel("test 准确率")
axes[0].set_title("树越多 → 收敛后平稳(不会过拟合)")

# max_features 影响去相关程度 / max_features
mfs = [1, 2, 3, 4, 5, 6]
mf_acc = [cross_val_score(RandomForestClassifier(200, max_features=m, random_state=0),X,y,cv=5).mean()
          for m in mfs]
axes[1].plot(mfs, mf_acc, "s-"); axes[1].axvline(np.sqrt(6), color="r", ls="--", label="√d 默认")
axes[1].set_xlabel("max_features"); axes[1].set_ylabel("CV 准确率"); axes[1].legend()
axes[1].set_title("max_features 小→树更去相关; 太小→单树太弱")
plt.tight_layout(); plt.show()
print("n_estimators 越多越好(只是变慢); max_features 控制去相关 vs 单树强度的权衡")


<a id="6"></a>
## 6. 特征重要性 + RF vs GBDT / Importance & vs GBDT

RF 的特征重要性 = 所有树上该特征带来的平均不纯度下降。**注意偏差**: 不纯度重要性偏向高基数/连续特征; 更可靠的是**置换重要性(permutation importance)**(打乱一列看性能掉多少, 3.x 提过)。


In [ ]:
from sklearn.inspection import permutation_importance
imp_gini = pd.Series(rf.feature_importances_, index=feat)
perm = permutation_importance(rf, X_te, y_te, n_repeats=20, random_state=0)
imp_perm = pd.Series(perm.importances_mean, index=feat)
cmp = pd.DataFrame({"不纯度重要性": imp_gini, "置换重要性": imp_perm}).sort_values("置换重要性", ascending=False)
print(cmp.round(3).to_string())
print("\nsex/pclass 最关键; 两种重要性大体一致但不完全相同(不纯度对连续特征 fare/age 略偏高)")


**RF vs GBDT(5.8)** —— 两大集成范式对比(面试高频):

| | 随机森林 (Bagging) | GBDT (Boosting) |
|---|---|---|
| 训练 | **并行**, 树相互独立 | **串行**, 每棵纠正前面的残差 |
| 目标 | 降**方差**(树本身低偏差高方差→深树) | 降**偏差**(弱树串联→浅树) |
| 过拟合 | 树多不会过拟合 | 树多**会**过拟合(需早停/学习率) |
| 调参 | 少, 开箱即用 | 多, 需细调 |
| 通常精度 | 强 | 调好后**更强**(表格之王) |


<a id="7"></a>
## 7. 小结 / Summary

```
随机森林 = bagging 树 + 特征随机; 投票降方差
bagging 数学: Var = ρσ² + (1-ρ)/B σ²; 加树消第二项, 但被 ρ 卡住 → 必须去相关
两处随机: 样本 bootstrap(~63%) + 每分裂随机 √d 个特征 → 降 ρ
OOB: ~37% 袋外样本做免费验证, ≈测试误差
树越多越稳(不过拟合); max_features 控去相关; 不纯度重要性偏连续特征→用置换重要性
RF(并行/降方差) vs GBDT(串行/降偏差)
```

### 💡 面试速查
1. **bagging 降方差**: 平均去相关预测; 公式 ρσ²+(1-ρ)σ²/B
2. **两处随机**: bootstrap 样本 + 随机特征子集; 特征随机是为**去相关**(光 bootstrap 不够)
3. **OOB** ≈ 免费的交叉验证(~37% 袋外)
4. **树多不过拟合**(与 GBDT 相反)
5. **不纯度重要性有偏** → 置换重要性更可靠

### 下一节
**5.8 GBDT**——从 bagging 切到 boosting。不再独立平均, 而是**一棵接一棵地纠正前面的错误**, 表格数据之王的起点。
